##
 1. Load data (dal CSV / parquet)
 2. Prezzi nel tempo (plot)
3. Distribuzione ritorni (istogrammi, QQ-plot)
 4. Correlation matrix (heatmap)
 5. VIX analysis
6. Rolling volatility
 7. Drawdown
 8. SPX/VIX rolling correlation

Prezzi nel tempo — plot multi-asset con eventi annotati (2008, COVID, 2022 energy crisis)
Distribuzione dei ritorni — istogrammi + QQ-plot vs normale, conferma visiva del Jarque-Bera
Correlation matrix — heatmap, magari separata per sotto-periodi (pre/post 2008)
VIX nel tempo — con soglie e regime flag sovrapposti
Volatility clustering — plot della rolling vol, visivamente si vede già la persistenza
Drawdown chart — per SPX e Brent principalmente
SPX/VIX rolling correlation — per mostrare che cambia nei regimi di stress
Analisi per sotto-periodi — statistiche separate per 2004-2008, 2009-2019, 2020-2022, 2023+

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import sys
import os
notebook_dir = os.getcwd()
parent_dir = os.path.dirname(notebook_dir)
sys.path.append(parent_dir)
from config_path import cfg_path

# Read processed data from the previously written csv


df = pd.read_parquet(os.path.join(cfg_path.workspace_root, cfg_path.processed_data))
df.head()


In [ ]:

print(df.shape)
df.describe()

## Price evolution over time 
I plot the historical price evolution of all assets in the basket (equity, energy, FX, VIX, Brent).

The goal is to visually identify the main market stress periods (2008 financial crisis, COVID-19 2020,

2022 energy crisis), which should later emerge as distinct regimes in the HMM and change point

detection models.

In [ ]:

price_cols = ["^GSPC", "FEZ", "CL=F", "NG=F", "EURUSD=X", "USDCHF=X", "^VIX", "Brent_EIA"]

fig = make_subplots(
    rows=4, cols=2,
    subplot_titles=price_cols
)

for i, col in enumerate(price_cols):
    row = i // 2 + 1
    col_pos = i % 2 + 1

    fig.add_trace(
        go.Scatter(x=df.index, y=df[col], name=col),
        row=row, col=col_pos
    )

fig.update_layout(height=900, width=1000, title="Prezzi storici per asset", showlegend=False)
fig.show()

## 2. VIX index over time

The VIX measures the market's expected 30-day volatility, implied by S&P 500 options prices.
It is mean-reverting and spikes sharply during market stress rather than trending persistently
like price levels. Two reference thresholds are shown: 20 (moderate volatility) and 30 (acute stress),
commonly used as informal regime boundaries in practitioner research.

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["^VIX"],
    name="VIX",
    line=dict(color="#5F5E5A", width=1.2),
    fill="tozeroy",
    fillcolor="rgba(95,94,90,0.05)"
))

fig.add_hline(y=20, line_dash="dash", line_color="#BA7517",
              annotation_text="20 — vol moderata", annotation_position="top left")
fig.add_hline(y=30, line_dash="dash", line_color="#A32D2D",
              annotation_text="30 — stress", annotation_position="top left")

fig.update_layout(
    title="VIX Index — 2004–2025",
    xaxis_title="Date",
    yaxis_title="VIX level",
    template="plotly_white",
    height=500, width=1000,
    showlegend=False
)

fig.show()

## 3. Log returns distribution

I examine the distributional properties of daily log returns, focusing on the S&P 500 as
the primary benchmark. The computed Jarque-Bera test
indicated departure from normality; here I visualize that departure directly through
a histogram with an overlaid normal density and a QQ-plot.

In [ ]:
import numpy as np
from scipy import stats
import statsmodels.api as sm
import matplotlib.pyplot as plt



col = "^GSPC_log_ret"
returns = df[col].dropna()

jb_stat, jb_pval = stats.jarque_bera(returns)
print(f"Jarque-Bera test — {col}")
print(f"Statistic: {jb_stat:.2f}, p-value: {jb_pval:.4g}")
print("Normal" if jb_pval > 0.05 else "Reject normality (non-normal distribution)")
col = "^GSPC_log_ret"
returns = df[col].dropna()

fig = go.Figure()

# histogram for log returns
fig.add_trace(go.Histogram(
    x=returns, histnorm="probability density", name="Log returns", nbinsx=100
))

# normal curve with the same mean and std as log returns
x_range = np.linspace(returns.min(), returns.max(), 200)
normal_curve = stats.norm.pdf(x_range, returns.mean(), returns.std())

fig.add_trace(go.Scatter(x=x_range, y=normal_curve, name="Normal distribution", line=dict(color="red")))

fig.update_layout(title=f"Distribution of log returns — {col}", template="plotly_white", height=500)
fig.show()


# QQ-plot for log returns
sm.qqplot(returns, line="s")
plt.title(f"QQ-plot — {col}")
plt.show()

## 5. Correlation matrix

I compute the pairwise correlation matrix of daily log returns across all assets.
This provides a static view of co-movement and helps sanity-check the dataset against
known relationships (e.g. positive correlation between S&P 500 and Euro Stoxx 50).

In [ ]:
log_ret_cols = [c for c in df.columns if c.endswith("_log_ret")]
corr_matrix = df[log_ret_cols].corr()

fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale="RdBu",
    zmid=0,
    text=corr_matrix.round(2).values,
    texttemplate="%{text}"
))

fig.update_layout(title="Correlation matrix — log returns", height=600, width=700)
fig.show()

## 6. Rolling volatility vs VIX

I plot the 21-day realized volatility of the S&P 500 alongside the VIX index.
Since the VIX is a forward-looking, options-implied measure while realized volatility is
backward-looking, I expect them to move together but not coincide exactly — divergences
between the two are themselves informative about market regime shifts.

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=df.index, y=df["^GSPC_vol_21d"],
    name="SPX realized vol (21d)",
    line=dict(color="#1D9E75")
))

fig.add_trace(go.Scatter(
    x=df.index, y=df["^VIX"],
    name="VIX",
    line=dict(color="#5F5E5A"),
    yaxis="y2"
))

fig.update_layout(
    title="Realized volatility (SPX, 21d) vs VIX",
    template="plotly_white",
    height=500,
    xaxis=dict(title="Date"),
    yaxis=dict(
        title=dict(text="SPX realized vol (21d)", font=dict(color="#1D9E75")),
        tickfont=dict(color="#1D9E75")
    ),
    yaxis2=dict(
        title=dict(text="VIX", font=dict(color="#5F5E5A")),
        tickfont=dict(color="#5F5E5A"),
        overlaying="y",
        side="right"
    ),
    legend=dict(x=0.01, y=0.99)
)

fig.show()

## 7. Drawdown

I compute the drawdown from peak for the S&P 500, i.e. the percentage decline from the
highest cumulative value reached so far. This highlights both the depth and duration of
major stress periods, complementing the volatility-based view above.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df["^GSPC_drawdown"], fill="tozeroy", name="SPX drawdown"))
fig.update_layout(title="SPX drawdown from peak", template="plotly_white", height=500)
fig.show()

## 8. Rolling correlation: S&P 500 vs VIX

I examine the 63-day rolling correlation between S&P 500 and VIX log returns. This
correlation is structurally negative under normal market conditions, but tends to
compress toward zero during periods of acute stress — making it a useful regime
indicator in its own right.

In [ ]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.index, y=df["spx_vix_rolling_corr"], name="SPX/VIX rolling corr (63d)"))
fig.add_hline(y=0, line_dash="dash", line_color="gray")
fig.update_layout(title="Rolling correlation — SPX vs VIX", template="plotly_white", height=500)
fig.show()

## 9. Sub-period statistics

I split the sample into five sub-periods (pre-GFC, the 2008-2009 financial crisis,
2010-2019, the 2020-2022 COVID/energy crisis window, and 2023-2025) and compute summary
statistics of S&P 500 log returns for each. This gives an empirical, model-free sense of
how many distinct regimes the data might contain, ahead of formal regime detection.

In [18]:
periods = {
    "2004-2007": ("2004-01-01", "2007-12-31"),
    "2008-2009 (GFC)": ("2008-01-01", "2009-12-31"),
    "2010-2019": ("2010-01-01", "2019-12-31"),
    "2020-2022 (COVID+energy)": ("2020-01-01", "2022-12-31"),
    "2023-2025": ("2023-01-01", "2025-12-31"),
}

records = []
for label, (start, end) in periods.items():
    subset = df.loc[start:end, "^GSPC_log_ret"].dropna()
    records.append({
        "period": label,
        "mean_daily_ret": subset.mean(),
        "ann_vol": subset.std() * np.sqrt(252),
        "skew": subset.skew(),
        "kurtosis": subset.kurt()
    })

pd.DataFrame(records).set_index("period")

,mean_daily_ret,ann_vol,skew,kurtosis
period,,,,
2004-2007,0.000272,0.121033,-0.317141,1.807681
2008-2009 (GFC),-0.000566,0.353882,-0.114471,4.203413
2010-2019,0.000427,0.148009,-0.551107,4.493974
2020-2022 (COVID+energy),0.000233,0.257371,-0.755528,10.849529
2023-2025,0.000794,0.150586,0.332480,14.339260


## 10. Stationarity check (Augmented Dickey-Fuller test)

Most time series models (HMM, GARCH) assume stationary input. I verify empirically what
is normally assumed: price levels should be non-stationary (they follow a random-walk-like
process with trend), while log returns should be stationary. The ADF test formalizes this —
a low p-value rejects the null hypothesis of a unit root, i.e. confirms stationarity.

In [19]:
from statsmodels.tsa.stattools import adfuller

def adf_test(series, name):
    result = adfuller(series.dropna())
    verdict = "stationary" if result[1] < 0.05 else "non-stationary"
    print(f"{name:25s} ADF stat={result[0]:8.3f}  p-value={result[1]:.4f}  -> {verdict}")

adf_test(df["^GSPC"], "SPX price level")
adf_test(df["^GSPC_log_ret"], "SPX log returns")
adf_test(df["^VIX"], "VIX level")
adf_test(df["Brent_EIA"], "Brent price level")

SPX price level           ADF stat=   2.548  p-value=0.9991  -> non-stationary
SPX log returns           ADF stat= -15.950  p-value=0.0000  -> stationary
VIX level                 ADF stat=  -5.796  p-value=0.0000  -> stationary
Brent price level         ADF stat=  -2.948  p-value=0.0400  -> stationary


## 11. Autocorrelation: returns vs squared returns

Market efficiency implies that raw returns should show little to no autocorrelation —
past returns shouldn't predict future ones. However, squared returns (a proxy for variance)
are expected to show strong, persistent autocorrelation. This is the empirical signature of
volatility clustering: large moves tend to be followed by large moves, regardless of sign.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(df["^GSPC_log_ret"].dropna(), lags=40, ax=axes[0])
axes[0].set_title("ACF — SPX log returns")
plot_acf(df["^GSPC_log_ret"].dropna()**2, lags=40, ax=axes[1])
axes[1].set_title("ACF — SPX squared log returns")
plt.tight_layout()
plt.show()

## 12. Ljung-Box test on squared returns

I formally test what the ACF plot above suggests visually: whether squared returns exhibit
statistically significant autocorrelation at multiple lags. A low p-value confirms the
presence of volatility clustering beyond what could be attributed to random noise.

In [21]:
from statsmodels.stats.diagnostic import acorr_ljungbox

lb_result = acorr_ljungbox(df["^GSPC_log_ret"].dropna()**2, lags=[5, 10, 20], return_df=True)
print(lb_result)

        lb_stat  lb_pvalue
5   3138.879570        0.0
10  5343.273677        0.0
20  7536.465902        0.0


## 13. ARCH test (conditional heteroskedasticity)

Engle's ARCH test directly examines whether the variance of returns is time-varying and
predictable from its own past — i.e. whether the series exhibits autoregressive conditional
heteroskedasticity. A significant result is the formal statistical justification for moving
to a GARCH-family model rather than assuming constant volatility.

In [22]:
from statsmodels.stats.diagnostic import het_arch

arch_stat, arch_pval, f_stat, f_pval = het_arch(df["^GSPC_log_ret"].dropna())
print(f"ARCH test (LM): stat={arch_stat:.2f}, p-value={arch_pval:.4f}")
print(f"ARCH test (F):  stat={f_stat:.2f}, p-value={f_pval:.4f}")

ARCH test (LM): stat=1543.86, p-value=0.0000
ARCH test (F):  stat=215.20, p-value=0.0000
